# Voltage Stability Analysis## ObjectiveAnalyzes voltage recovery and reactive power support during faultsProject: 39 Bus New England System - 2**Study Case: Study Cases 1. Power Flow****Objective:**- Analyze voltage recovery and reactive power support during faults- Apply a single-phase fault followed by successful auto-reclosing- Simulate both scenarios- Monitor reactive power responses- Outputs: Reactive power vs. time, Voltage recovery curves, Comparative analysis---

## Step 1: Access PowerFactoryFirst, we need to set up the Python environment to access DIgSILENT PowerFactory.

In [ ]:
# ============================================================================# STEP 1: Access PowerFactory# ============================================================================import osos.environ["PATH"] = r"C:\Program Files\DIgSILENT\PowerFactory 2021 SP2" + os.environ["PATH"]import syssys.path.append(r"C:\Program Files\DIgSILENT\PowerFactory 2021 SP2\Python\3.9")# Import powerfactoryimport powerfactory as pfapp = pf.GetApplication()  # Get the application# ============================================================================# STEP 2: Access and activate project# ============================================================================user = app.GetCurrentUser()project = app.ActivateProject("39 Bus New England System - 2")  # Activate the desired projectprj = app.GetActiveProject()print(f"Project activated: {prj.loc_name}")# ============================================================================# STEP 2.5: Activate study case (if needed)# ============================================================================# Try to activate the study case "Study Cases 1. Power Flow"try:    study_cases = prj.GetContents('*.IntCase')    for sc in study_cases:        if '1. Power Flow' in sc.loc_name or 'Power Flow' in sc.loc_name:            sc.Activate()            print(f"Study case activated: {sc.loc_name}")            breakexcept:    print("Note: Using default/active study case")# ============================================================================# STEP 3: Get all relevant objects (buses, generators, lines)# ============================================================================# Create bus dictionarybuses = app.GetCalcRelevantObjects('*.ElmTerm')bus_dict = {}for bus in buses:    bus_dict[bus.loc_name] = bus# Create generator dictionarygenerators = app.GetCalcRelevantObjects('*.ElmSym')gen_dict = {}for gen in generators:    gen_dict[gen.loc_name] = gen# Create line dictionarylines = app.GetCalcRelevantObjects('*.ElmLne')line_dict = {}for line in lines:    line_dict[line.loc_name] = lineprint(f"Found {len(bus_dict)} buses, {len(gen_dict)} generators, and {len(line_dict)} lines")# ============================================================================# STEP 4: Select fault location and line for auto-reclosing - Base Case# ============================================================================print("\n=== Setting up Single-Phase Fault with Auto-Reclosing - Base Case ===")app.ResetCalculation()# Select a line for fault and auto-reclosingfault_line = Noneif len(line_dict) > 0:    fault_line = list(line_dict.values())[0]    print(f"Selected fault location: {fault_line.loc_name}")# Select a bus for monitoringmonitor_bus = Noneif 'Bus 16' in bus_dict:    monitor_bus = bus_dict['Bus 16']elif len(bus_dict) > 0:    monitor_bus = list(bus_dict.values())[0]if fault_line and monitor_bus:    # Create single-phase fault event    event_folder = app.GetFromStudyCase('IntEvt')    event_name = 'Single Phase Fault Base'        try:        # Create fault event        event_folder.CreateObject('EvtShc', event_name)        fault_event = event_folder.GetContents(event_name)[0] if hasattr(event_folder, 'GetContents') else None                if fault_event is None:            events = event_folder.GetContents()            fault_event = events[0] if events else None                if fault_event:            fault_event.time = 0.5  # Fault occurs at 0.5 seconds            fault_event.p_target = monitor_bus  # Fault at bus            fault_event.i_shc = 1  # Single-phase fault (phase A)            fault_event.t_clear = 0.1  # Fault clearing time (100 ms)                        print(f"Single-phase fault created at {monitor_bus.loc_name}")            print(f"Fault time: {fault_event.time} s, Clearing time: {fault_event.t_clear} s")                        # Note: Auto-reclosing would typically be configured in protection settings            # This is a simplified representation    except Exception as e:        print(f"Note: Fault event creation may need manual setup: {e}")# ============================================================================# STEP 5: Set up results monitoring - Base Case# ============================================================================elmres = app.GetFromStudyCase('All calculations.ElmRes')elmres.Clear()# Add bus voltage monitoringif monitor_bus:    elmres.AddVariable(monitor_bus, 'm:u')  # Voltage magnitude# Add reactive power monitoring for generatorsfor gen_name, gen in gen_dict.items():    try:        elmres.AddVariable(gen, 'm:Q:bus1')  # Reactive power output    except:        pass# Add reactive power monitoring for linesif fault_line:    try:        elmres.AddVariable(fault_line, 'm:Q:bus1')  # Reactive power flow        elmres.AddVariable(fault_line, 'm:Q:bus2')    except:        passprint("Added monitoring for voltage and reactive power")# ============================================================================# STEP 6: Run Dynamic Simulation - Base Case# ============================================================================print("\n=== Running Dynamic Simulation - Base Case ===")# Set initial conditionsini = app.GetFromStudyCase('ComInc')ini.Execute()print("Initial conditions calculated")# Run dynamic simulationsim = app.GetFromStudyCase('ComSim')sim.tstop = 5.0  # Simulation time: 5 secondssim.Execute()print(f"Dynamic simulation completed (t = 0 to {sim.tstop} s)")# ============================================================================# STEP 7: Export Base Case Results# ============================================================================import osscript_dir = os.path.dirname(os.path.abspath(__file__))comres = app.GetFromStudyCase('ComRes')comres.iopt_csel = 0comres.iopt_locn = 1comres.ciopt_head = 1comres.pResult = elmrescomres.ipt_exp = 6  # CSV filecomres.f_name = os.path.join(script_dir, 'voltage_stability_base_case.csv')comres.Execute()print(f"Base case results exported to: voltage_stability_base_case.csv")# ============================================================================# STEP 8: New Generation Case Voltage Stability# ============================================================================print("\n=== Setting up Single-Phase Fault with Auto-Reclosing - New Generation Case ===")# NOTE: This section should be modified based on how new generation is addedapp.ResetCalculation()# Re-create fault event for new generation caseif fault_line and monitor_bus:    event_folder = app.GetFromStudyCase('IntEvt')    event_name_new = 'Single Phase Fault New Gen'        try:        event_folder.CreateObject('EvtShc', event_name_new)        fault_event_new = event_folder.GetContents(event_name_new)[0] if hasattr(event_folder, 'GetContents') else None                if fault_event_new is None:            events = event_folder.GetContents()            fault_event_new = events[-1] if events else None                if fault_event_new:            fault_event_new.time = 0.5            fault_event_new.p_target = monitor_bus            fault_event_new.i_shc = 1            fault_event_new.t_clear = 0.1                        print(f"Single-phase fault created for New Generation Case")    except Exception as e:        print(f"Note: Fault event creation may need manual setup: {e}")# Set up results monitoringelmres.Clear()if monitor_bus:    elmres.AddVariable(monitor_bus, 'm:u')for gen_name, gen in gen_dict.items():    try:        elmres.AddVariable(gen, 'm:Q:bus1')    except:        passif fault_line:    try:        elmres.AddVariable(fault_line, 'm:Q:bus1')        elmres.AddVariable(fault_line, 'm:Q:bus2')    except:        pass# Set initial conditionsini.Execute()# Run dynamic simulationsim.tstop = 5.0sim.Execute()print(f"Dynamic simulation completed for New Generation Case (t = 0 to {sim.tstop} s)")# ============================================================================# STEP 9: Export New Generation Case Results# ============================================================================comres.f_name = os.path.join(script_dir, 'voltage_stability_new_gen_case.csv')comres.Execute()print(f"New generation case results exported to: voltage_stability_new_gen_case.csv")# ============================================================================# STEP 10: Load CSV Data and Create Visualizations# ============================================================================print("\n=== Creating Visualizations ===")try:    import pandas as pd    import matplotlib.pyplot as plt    import seaborn as sns    from bokeh.plotting import figure, output_file, save    from bokeh.models import ColumnDataSource, HoverTool    from bokeh.layouts import gridplot        # Set style    sns.set_style("whitegrid")    plt.rcParams['figure.figsize'] = (16, 10)        # Read CSV files    base_df = pd.read_csv(os.path.join(script_dir, 'voltage_stability_base_case.csv'))    new_gen_df = pd.read_csv(os.path.join(script_dir, 'voltage_stability_new_gen_case.csv'))        # Get time column    time_col = base_df.columns[0]        # Find relevant columns    voltage_cols_base = [col for col in base_df.columns if 'u' in col.lower() or 'voltage' in col.lower()]    voltage_cols_new = [col for col in new_gen_df.columns if 'u' in col.lower() or 'voltage' in col.lower()]    q_cols_base = [col for col in base_df.columns if 'Q' in col or 'reactive' in col.lower()]    q_cols_new = [col for col in new_gen_df.columns if 'Q' in col or 'reactive' in col.lower()]        # Create comprehensive plots    num_subplots = sum([len(voltage_cols_base) > 0, len(q_cols_base) > 0])    if num_subplots == 0:        num_subplots = 1        fig, axes = plt.subplots(num_subplots, 1, figsize=(14, 5 * num_subplots))    if num_subplots == 1:        axes = [axes]        plot_idx = 0        # Plot voltage recovery    if voltage_cols_base and voltage_cols_new and plot_idx < len(axes):        axes[plot_idx].plot(base_df[time_col], base_df[voltage_cols_base[0]],                           label='Base Case', color='blue', linewidth=2)        axes[plot_idx].plot(new_gen_df[time_col], new_gen_df[voltage_cols_new[0]],                           label='New Generation Case', color='red', linewidth=2)        axes[plot_idx].axhline(y=1.0, color='black', linestyle='--', linewidth=1, alpha=0.5, label='Nominal (1.0 p.u.)')        axes[plot_idx].set_xlabel('Time (s)', fontsize=12)        axes[plot_idx].set_ylabel('Voltage (p.u.)', fontsize=12)        axes[plot_idx].set_title(f'Voltage Recovery Curve: {monitor_bus.loc_name if monitor_bus else "Bus"}',                                 fontsize=12, fontweight='bold')        axes[plot_idx].legend()        axes[plot_idx].grid(True, alpha=0.3)        plot_idx += 1        # Plot reactive power responses    if q_cols_base and q_cols_new and plot_idx < len(axes):        for col_base, col_new in zip(q_cols_base[:3], q_cols_new[:3]):            axes[plot_idx].plot(base_df[time_col], base_df[col_base],                                linewidth=1.5, alpha=0.7, label=f'Base: {col_base}')            axes[plot_idx].plot(new_gen_df[time_col], new_gen_df[col_new],                                '--', linewidth=1.5, alpha=0.7, label=f'New Gen: {col_new}')        axes[plot_idx].set_xlabel('Time (s)', fontsize=12)        axes[plot_idx].set_ylabel('Reactive Power (Mvar)', fontsize=12)        axes[plot_idx].set_title('Reactive Power Response', fontsize=12, fontweight='bold')        axes[plot_idx].legend(fontsize=8)        axes[plot_idx].grid(True, alpha=0.3)        plt.tight_layout()    plot_path = os.path.join(script_dir, 'voltage_stability_plots.png')    plt.savefig(plot_path, dpi=300, bbox_inches='tight')    plt.close()    print(f"Static plots saved to: voltage_stability_plots.png")        # Interactive Bokeh plots    try:        output_file(os.path.join(script_dir, 'voltage_stability_interactive.html'))        plots = []                # Voltage plot        if voltage_cols_base and voltage_cols_new:            p1 = figure(width=900, height=400, title="Voltage Recovery (Interactive)",                       x_axis_label="Time (s)", y_axis_label="Voltage (p.u.)",                       tools="pan,wheel_zoom,box_zoom,reset,hover,save")                        base_source = ColumnDataSource(data=dict(                time=base_df[time_col],                voltage=base_df[voltage_cols_base[0]]            ))                        new_gen_source = ColumnDataSource(data=dict(                time=new_gen_df[time_col],                voltage=new_gen_df[voltage_cols_new[0]]            ))                        p1.line('time', 'voltage', source=base_source, color='blue',                    line_width=2, legend_label='Base Case', alpha=0.8)            p1.line('time', 'voltage', source=new_gen_source, color='red',                    line_width=2, legend_label='New Generation Case', alpha=0.8)            p1.line([base_df[time_col].min(), base_df[time_col].max()], [1.0, 1.0],                    color='black', line_dash='dashed', line_width=1, legend_label='Nominal')                        hover = p1.select_one(HoverTool)            hover.tooltips = [("Time", "@time{0.00} s"), ("Voltage", "@voltage{0.000} p.u.")]            p1.legend.location = "top_right"            plots.append(p1)                # Reactive power plot        if q_cols_base and q_cols_new:            p2 = figure(width=900, height=400, title="Reactive Power Response (Interactive)",                       x_axis_label="Time (s)", y_axis_label="Reactive Power (Mvar)",                       tools="pan,wheel_zoom,box_zoom,reset,hover,save")                        for col_base, col_new in zip(q_cols_base[:3], q_cols_new[:3]):                base_src = ColumnDataSource(data=dict(time=base_df[time_col], q=base_df[col_base]))                new_gen_src = ColumnDataSource(data=dict(time=new_gen_df[time_col], q=new_gen_df[col_new]))                p2.line('time', 'q', source=base_src, line_width=2, alpha=0.7, legend_label=f'Base: {col_base}')                p2.line('time', 'q', source=new_gen_src, line_width=2, alpha=0.7,                        line_dash='dashed', legend_label=f'New Gen: {col_new}')                        hover2 = p2.select_one(HoverTool)            hover2.tooltips = [("Time", "@time{0.00} s"), ("Q", "@q{0.0} Mvar")]            p2.legend.location = "top_right"            plots.append(p2)                if plots:            grid = gridplot([plots], toolbar_location='right')            save(grid)            print(f"Interactive plots saved to: voltage_stability_interactive.html")    except Exception as e:        print(f"Note: Bokeh interactive plot creation failed: {e}")    except ImportError as e:    print(f"Note: Visualization libraries not available: {e}")    print("Install required packages: pip install matplotlib seaborn pandas bokeh")except Exception as e:    print(f"Note: Error creating plots: {e}")# ============================================================================# STEP 11: Clean up# ============================================================================app.ResetCalculation()# Delete eventstry:    event_folder = app.GetFromStudyCase('IntEvt')    events = event_folder.GetContents()    for event in events:        if 'Single Phase Fault' in event.loc_name:            event.Delete()except:    passprint("\n=== Voltage Stability Analysis completed successfully ===")print(f"Results saved in: {script_dir}")